# Documentação — Notebook 01_conexao_adls_gen2

Este notebook faz a **conexão com o Azure Data Lake Storage Gen2** e lê os arquivos Parquet da tabela `ecommerce_clientes` diretamente do container `raw`.

**⚠️ Atenção:** Este notebook usa Azure SDK, que **não funciona no compute serverless**. Funciona apenas em cluster clássico. No serverless, os dados são acessados via Lakehouse Federation (`sqlserver_catalog.squad2.ecommerce_clientes`).

## Célula 1 — Instalação de bibliotecas

```python
%pip install azure-storage-file-datalake azure-identity pandas pyarrow
dbutils.library.restartPython()
```

**O que faz:** Instala as bibliotecas do Azure SDK necessárias para acessar o ADLS Gen2.
- `azure-storage-file-datalake` — cliente do Data Lake Storage
- `azure-identity` — autenticação com o Azure (Service Principal)
- `pandas` e `pyarrow` — leitura de arquivos Parquet

`dbutils.library.restartPython()` reinicia o kernel Python para garantir que as libs instaladas fiquem disponíveis.

## Célula 2 — Importação de bibliotecas

```python
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient
import pandas as pd
import os
```

**O que faz:** Importa as classes do Azure SDK:
- `ClientSecretCredential` — autentica no Azure usando Tenant ID, Client ID e Client Secret
- `DataLakeServiceClient` — cliente principal para acessar o ADLS Gen2
- `pandas` — manipulação local dos dados Parquet
- `os` — acessar variáveis de ambiente do `.env`

## Célula 3 — Carregamento do .env

```python
env_path = next(Path("/Workspace").rglob(".env"), None)
load_dotenv(env_path)
```

**O que faz:** Busca e carrega o arquivo `.env` com as credenciais (mesmo padrão do `00_setup_config`). Necessário para obter `ADLS_TENANT_ID`, `ADLS_CLIENT_ID` e `ADLS_CLIENT_SECRET`.

## Célula 4 — Criação da credencial Azure

```python
credential = ClientSecretCredential(
    tenant_id=os.getenv("ADLS_TENANT_ID"),
    client_id=os.getenv("ADLS_CLIENT_ID"),
    client_secret=os.getenv("ADLS_CLIENT_SECRET")
)
```

**O que faz:** Cria a credencial de autenticação com o Azure usando o fluxo **Service Principal** (Client Credentials). Esta credencial é usada para todas as operações subsequentes no ADLS.

**Como funciona:** O Azure valida o trio (Tenant ID, Client ID, Client Secret) e retorna um token de acesso OAuth 2.0 que permite ler/escrever no Data Lake.

## Célula 5 — Conexão com o ADLS Gen2

```python
storage_account_name = "internshipdatalake"
container_name = "raw"

service_client = DataLakeServiceClient(
    account_url=f"https://{storage_account_name}.dfs.core.windows.net",
    credential=credential
)
fs_client = service_client.get_file_system_client(container_name)
```

**O que faz:** Cria dois clientes:
- `service_client` — conexão com a conta de armazenamento `internshipdatalake`
- `fs_client` — conexão com o container `raw` (onde estão os dados em tempo real)

A partir daqui, `fs_client` pode listar, ler e baixar arquivos do container.

## Célula 6 — Listagem de arquivos Parquet

```python
for arquivo in fs_client.get_paths(recursive=True):
    if arquivo.name.endswith("ecommerce_clientes.parquet"):
        arquivos_encontrados.append(arquivo.name)
```

**O que faz:** Percorre recursivamente todos os caminhos do container `raw` e filtra apenas os arquivos que terminam com `ecommerce_clientes.parquet`.

**Resultado esperado:** Lista de arquivos no formato `real-time-data/2026/09/21/112245/ecommerce_clientes.parquet` — os micro-lotes gerados em tempo real.

## Célula 7 — Leitura e junção dos Parquets

```python
for caminho in arquivos_encontrados:
    conteudo = fs_client.get_file_client(caminho).download_file().readall()
    df_temp = pd.read_parquet(BytesIO(conteudo))
    dfs_clientes.append(df_temp)

df_clientes_pd = pd.concat(dfs_clientes, ignore_index=True)
```

**O que faz:** Para cada arquivo encontrado:
1. Baixa o conteúdo binário do Parquet do ADLS
2. Converte para um pandas DataFrame
3. Acumula em uma lista

No final, concatena todos os DataFrames em um único pandas DataFrame.

**Resultado esperado:** ~384 linhas, 8 colunas.

## Célula 8 — Conversão para PySpark

```python
df_clientes = spark.createDataFrame(df_clientes_pd)
```

**O que faz:** Converte o pandas DataFrame para um PySpark DataFrame. Isso permite usar as operações distribuídas do Spark nos dados lidos do ADLS.

**Resultado esperado:** `Linhas: 384, Colunas: 8`

A partir daqui, `df_clientes` está disponível para os próximos notebooks (como `02_data_load_sqlserver` que faz `%run ./01_conexao_adls_gen2`).

## Resumo do fluxo

| Célula | Função |
| --- | --- |
| 1 | Instala Azure SDK e reinicia Python |
| 2 | Importa ClientSecretCredential e DataLakeServiceClient |
| 3 | Carrega .env com credenciais |
| 4 | Cria credencial Azure (Service Principal) |
| 5 | Conecta ao ADLS Gen2 (container raw) |
| 6 | Lista arquivos ecommerce_clientes.parquet |
| 7 | Baixa e junta todos os Parquets em pandas DF |
| 8 | Converte para PySpark DataFrame |

**Fluxo:** Instala → autentica → conecta → lista → lê → converte

**⚠️ Limitação serverless:** Este notebook não funciona no serverless pois o Azure SDK requer acesso de rede ao Azure. No serverless, use Lakehouse Federation (`sqlserver_catalog.squad2.ecommerce_clientes`).